# Đánh giá mức độ ảnh hưởng khi loại bỏ Campaign TypeA

Mục đích của Notebook này là thống kê lượng dữ liệu (campaign, coupon, redeem, household) còn lại nếu chúng ta quyết định loại bỏ hoàn toàn các Campaign `TypeA` do hạn chế về dữ liệu (không biết khách hàng được phân bổ chính xác 16 coupon nào).

In [1]:
import pandas as pd
from pathlib import Path

# Tìm thư mục chứa dữ liệu
SEARCH_ROOTS = [
    Path("./data"),
    Path("./dataset"),
    Path("./The Complete Journey"),
    Path("."),
]

csv_paths = {}
for root in SEARCH_ROOTS:
    if root.exists():
        for p in root.rglob("*.csv"):
            csv_paths[p.stem] = p

print("Đang tải dữ liệu...")
campaign_desc = pd.read_csv(csv_paths['campaign_desc'])
campaign_table = pd.read_csv(csv_paths['campaign_table'])
coupon = pd.read_csv(csv_paths['coupon'])
coupon_redempt = pd.read_csv(csv_paths['coupon_redempt'])
transaction_data = pd.read_csv(csv_paths['transaction_data'])
print("Tải dữ liệu hoàn tất!")

Đang tải dữ liệu...
Tải dữ liệu hoàn tất!


In [2]:
# 1. Thống kê Campaign Type
print("Phân bố các loại Campaign:")
print(campaign_desc['DESCRIPTION'].value_counts())

# Lấy ID của các campaign TypeA và Non-TypeA
type_a_campaigns = campaign_desc[campaign_desc['DESCRIPTION'] == 'TypeA']['CAMPAIGN'].tolist()
non_type_a_campaigns = campaign_desc[campaign_desc['DESCRIPTION'] != 'TypeA']['CAMPAIGN'].tolist()

print(f"\nSố lượng Campaign TypeA: {len(type_a_campaigns)}")
print(f"Số lượng Campaign TypeB & TypeC: {len(non_type_a_campaigns)}")

Phân bố các loại Campaign:
DESCRIPTION
TypeB    19
TypeC     6
TypeA     5
Name: count, dtype: int64

Số lượng Campaign TypeA: 5
Số lượng Campaign TypeB & TypeC: 25


In [3]:
# 2. Ảnh hưởng lên bảng campaign_table (Hộ gia đình nhận campaign)
total_campaign_table = len(campaign_table)
type_a_campaign_table = len(campaign_table[campaign_table['CAMPAIGN'].isin(type_a_campaigns)])
non_type_a_campaign_table = len(campaign_table[campaign_table['CAMPAIGN'].isin(non_type_a_campaigns)])

print(f"--- Bảng campaign_table (Lượt hộ gia đình nhận chiến dịch) ---")
print(f"- Tổng số lượt ban đầu: {total_campaign_table:,}")
print(f"- Số lượt còn lại (Non-TypeA): {non_type_a_campaign_table:,} ({(non_type_a_campaign_table/total_campaign_table)*100:.2f}%)")
print(f"- Số lượt bị loại bỏ (TypeA): {type_a_campaign_table:,} ({(type_a_campaign_table/total_campaign_table)*100:.2f}%)")

# Xem lượng hộ gia đình (household_key) còn lại
total_households = campaign_table['household_key'].nunique()
non_type_a_households = campaign_table[campaign_table['CAMPAIGN'].isin(non_type_a_campaigns)]['household_key'].nunique()
print(f"\n- Tổng số hộ gia đình có tham gia chiến dịch: {total_households:,}")
print(f"- Số hộ gia đình tham gia chiến dịch TypeB & C: {non_type_a_households:,} ({(non_type_a_households/total_households)*100:.2f}%)")

--- Bảng campaign_table (Lượt hộ gia đình nhận chiến dịch) ---
- Tổng số lượt ban đầu: 7,208
- Số lượt còn lại (Non-TypeA): 3,229 (44.80%)
- Số lượt bị loại bỏ (TypeA): 3,979 (55.20%)

- Tổng số hộ gia đình có tham gia chiến dịch: 1,584
- Số hộ gia đình tham gia chiến dịch TypeB & C: 1,076 (67.93%)


In [4]:
# 3. Ảnh hưởng lên bảng coupon (Danh sách phát hành)
total_coupon = len(coupon)
non_type_a_coupon = len(coupon[coupon['CAMPAIGN'].isin(non_type_a_campaigns)])

print(f"--- Bảng coupon (Danh sách coupon phát hành) ---")
print(f"- Tổng số dòng ban đầu: {total_coupon:,}")
print(f"- Số dòng còn lại (Non-TypeA): {non_type_a_coupon:,} ({(non_type_a_coupon/total_coupon)*100:.2f}%)")

--- Bảng coupon (Danh sách coupon phát hành) ---
- Tổng số dòng ban đầu: 124,548
- Số dòng còn lại (Non-TypeA): 21,655 (17.39%)


In [5]:
# 4. Ảnh hưởng lên bảng coupon_redempt (Lượt sử dụng thực tế)
total_redempt = len(coupon_redempt)
non_type_a_redempt = len(coupon_redempt[coupon_redempt['CAMPAIGN'].isin(non_type_a_campaigns)])

print(f"--- Bảng coupon_redempt (Lượt sử dụng coupon) ---")
print(f"- Tổng số lượt redeem ban đầu: {total_redempt:,}")
print(f"- Số lượt redeem còn lại (Non-TypeA): {non_type_a_redempt:,} ({(non_type_a_redempt/total_redempt)*100:.2f}%)")

--- Bảng coupon_redempt (Lượt sử dụng coupon) ---
- Tổng số lượt redeem ban đầu: 2,318
- Số lượt redeem còn lại (Non-TypeA): 527 (22.74%)


In [7]:
# 5. Ảnh hưởng lên transaction_data
tx_with_coupon = transaction_data[transaction_data['COUPON_DISC'] < 0]

print(f"--- Bảng transaction_data ---")
print(f"- Tổng số giao dịch (tất cả): {len(transaction_data):,}")
print(f"- Tổng số giao dịch có dùng coupon (COUPON_DISC < 0): {len(tx_with_coupon):,}")
print("\nLƯU Ý QUAN TRỌNG VỀ CAUSAL ML:")
print("Trong mô hình học máy nhân quả (Uplift Modeling), chúng ta KHÔNG xóa bỏ các giao dịch không thuộc chiến dịch (hoặc các giao dịch thuộc TypeA khỏi bảng transaction).")
print("Các giao dịch không nhận/sử dụng coupon chính là dữ liệu nhóm kiểm soát (Control Group) hoặc dữ liệu baseline để đối chiếu mức độ tăng doanh thu (Outcome).")
print("Việc loại bỏ TypeA chỉ có nghĩa là chúng ta không gán nhãn Treatment = 1 cho các sự kiện thuộc TypeA, và chỉ dùng TypeB & C làm nhóm thử nghiệm (Treatment Group).")

--- Bảng transaction_data ---
- Tổng số giao dịch (tất cả): 2,595,732
- Tổng số giao dịch có dùng coupon (COUPON_DISC < 0): 36,422

LƯU Ý QUAN TRỌNG VỀ CAUSAL ML:
Trong mô hình học máy nhân quả (Uplift Modeling), chúng ta KHÔNG xóa bỏ các giao dịch không thuộc chiến dịch (hoặc các giao dịch thuộc TypeA khỏi bảng transaction).
Các giao dịch không nhận/sử dụng coupon chính là dữ liệu nhóm kiểm soát (Control Group) hoặc dữ liệu baseline để đối chiếu mức độ tăng doanh thu (Outcome).
Việc loại bỏ TypeA chỉ có nghĩa là chúng ta không gán nhãn Treatment = 1 cho các sự kiện thuộc TypeA, và chỉ dùng TypeB & C làm nhóm thử nghiệm (Treatment Group).
